# ĐỒ ÁN TOÁN ỨNG DỤNG VÀ THỐNG KÊ
# PHẦN 1: PHÉP KHỬ GAUSS VÀ CÁC ỨNG DỤNG

- **Nhóm thực hiện:** Nhóm 14 - Lớp 24CTT2
- **Mục tiêu:** Cài đặt phép khử Gauss có partial pivoting từ đầu bằng Python để giải hệ phương trình tuyến tính, tính định thức, tìm ma trận nghịch đảo, tính hạng và tìm cơ sở.

**0.  Khai báo thư viện và hàm in ma trận**

In [1]:
import numpy as np
from IPython.display import display, Math
def print_matrix(title, matrix):
    """Hàm hỗ trợ in ma trận và vector căn lề chuẩn xác, xử lý số cực nhỏ"""
    print(f"\n--- {title} ---")
    
    # Nếu không có dữ liệu
    if matrix is None:
        print("   (Không tồn tại)")
        return
        
    # Xử lý nếu nó là Vector 1 chiều (ví dụ: b_eq = [14, 32, 50])
    if isinstance(matrix, list) and len(matrix) > 0 and not isinstance(matrix[0], list):
        # Làm tròn, nhưng nếu số quá nhỏ (gần bằng 0) thì in ra 0.0 luôn cho đẹp
        clean_vector = [val if abs(val) > 1e-10 else 0.0 for val in matrix]
        print("   [" + " ".join(f"{round(val, 4):>8}" for val in clean_vector) + "]")
        return

    # 3. Xử lý nếu nó là Ma trận 2 chiều
    for row in matrix:
        clean_row = [val if abs(val) > 1e-10 else 0.0 for val in row]
        print("   [" + " ".join(f"{round(val, 4):>8}" for val in clean_row) + "]")



**1.  Giải hệ phương trình bằng phép khử Gauss với Partial Pivoting**

In [ ]:
def back_substitution(U, c):
    """
    Thực hiện phép thế ngược để giải hệ tam giác trên
    
    Args:
        U: Ma trận tam giác trên
        c: Vector vế phải
    
    Returns:
        Vector nghiệm x
    """
    n = len(U)
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        # Kiểm tra phần tử trên đường chéo chính
        if abs(U[i][i]) < 1e-12:
            return None
        x[i] = (c[i] - sum(U[i][j] * x[j] for j in range(i + 1, n))) / U[i][i]
    return x

def gaussian_eliminate(A, b):
    """
    Thực hiện phép biến đổi Gauss để đưa ma trận A về dạng tam giác trên
    
    Args:
        A: Ma trận hệ số
        b: Vector vế phải
    
    Returns:
        Ma trận sau khi khử, nghiệm x, số lần hoán đổi
    """
    # Ghép ma trận A và vector b thành ma trận tăng cường M
    M = [row + [b[i]] for i, row in enumerate(A)]
    row = len(M)
    col = len(M[0])

    s = 0  # Đếm số lần hoán đổi dòng
    EPSILON = 1e-12
    pivot_cols = []   
    current_row = 0

    for k in range(col-1):
        if current_row >= row:
            break

        # Tìm phần tử chốt (trị tuyệt đối lớn nhất) để giảm sai số
        p = current_row
        for i in range(current_row + 1, row):
            if abs(M[i][k]) > abs(M[p][k]):
                p = i

        if abs(M[p][k]) < EPSILON: 
            print(f"Không có pivot tại cột {k}")
            continue
        
        if p != current_row:
            M[current_row], M[p] = M[p], M[current_row]
            s += 1
        pivot_cols.append(k)
        for i in range(current_row + 1, row):
            l_ik = M[i][k] / M[current_row][k]
            M[i][k] = 0
            for j in range(k + 1, col):
                M[i][j] -= l_ik * M[current_row][j]
        current_row += 1

    rank = len(pivot_cols)  #Tính hạng của ma trận

    #Hệ vô nghiệm
    #Tồn tại dòng có vế trái bằng 0 nhưng vế phải khác 0
    for i in range(rank, row):
        if abs(M[i][col-1]) > EPSILON:
            raise ValueError("Hệ phương trình vô nghiệm.")
    
    # Hệ có vô số nghiệm
    # rank < n (số ẩn): hệ vố số nghiệm
    if rank < col - 1:
        free_cols = [j for j in range(col-1) if j not in pivot_cols]

        # 1. Tìm nghiệm riêng x_p (Ngầm định các ẩn tự do = 0)
        x_p = [0.0] * (col-1)
        for i in range(rank - 1, -1, -1):
            p_col = pivot_cols[i]

            #Tính tổng giá trị của tất cả các ẩn số nằm bên phải ẩn chốt hiện tại
            s_val = sum(M[i][j] * x_p[j] for j in range(p_col + 1, col-1))  
            # x_chốt = (vế phải - tổng đã chuyển vế) / hệ số chốt
            x_p[p_col] = (M[i][col-1] - s_val) / M[i][p_col] 

        # 2. Tìm cơ sở không gian nghiệm (Giải hệ thuần nhất Ax = 0, bật lần lượt ẩn tự do = 1)
        null_basis = []
        for f in free_cols:
            v = [0.0] * (col-1)
            v[f] = 1.0  # Lần lượt cho từng ẩn tự do bằng 1, các ẩn tự do khác bằng 0
            for i in range(rank - 1, -1, -1):
                p_col = pivot_cols[i]

                # Tính tổng các ẩn nằm bên phải ẩn chốt hiện tại
                s_val = sum(M[i][j] * v[j] for j in range(p_col + 1, col-1))

                # Tìm ẩn chốt: x_chốt = -tổng / hệ số chốt (Do 0 - s_val = -s_val)
                v[p_col] = -s_val / M[i][p_col]
            null_basis.append(v)

        # 3. Nghiệm tổng quát = Nghiệm riêng + các cơ sở không gian nghiệm (kèm tham số c)
        formula = f"x = {[round(val, 4) for val in x_p]}"
        for idx, v in enumerate(null_basis):
            formula += f" + c{idx+1}*{[round(val, 4) for val in v]}"
        print("\nHệ có vô số nghiệm, công thức nghiệm tổng quát:")
        print(formula)
        return M, formula, s
    
    #Hệ có nghiệm duy nhất
    #rank = n (số ẩn)
    U = [row[:-1] for row in M[:rank]]
    c = [row[-1] for row in M[:rank]]
    x = back_substitution(U, c)

    return M, x, s 

def verify_solution(A, b, x_custom):
    """
    Kiểm chứng kết quả bằng NumPy

    Args:
        A: Ma trận hệ số
        b: Vector vế phải
        x_custom: nghiệm
    
    Return:
        True: Kết quả của bạn Khớp
        False: Kết quả của bạn Sai

    """
    import numpy as np
    # Xử lý kiểm tra cho trường hợp hệ vô số nghiệm / vô nghiệm (x là chuỗi hoặc x = None)
    if isinstance(x_custom, str) or x_custom is None:
        try:
            np.linalg.solve(np.array(A, dtype=float), np.array(b, dtype=float))
            return False 
        except (np.linalg.LinAlgError, ValueError):
            return True 
        
    # Dùng numpy để kiểm tra lại trường hợp có nghiệm duy nhất
    A_np = np.array(A, dtype=float)
    b_np = np.array(b, dtype=float)
    x_np = np.array(x_custom, dtype=float)

    # Kiểm tra xem A * x có xấp xỉ bằng b không
    return np.allclose(np.dot(A_np, x_np), b_np)



- Kiểm thử và kiểm chứng

In [ ]:
print("KỊCH BẢN 1: HỆ PHƯƠNG TRÌNH CÓ NGHIỆM DUY NHẤT")

A_eq = [[1, 2, 3], 
        [0, 1, 4], 
        [5, 6, 0]]
b_eq = [14, 32, 50]

print_matrix("Ma trận hệ số gốc A", A_eq)
print_matrix("Vector vế phải ban đầu b", b_eq)
M_res_eq, sol_eq, swaps_eq = gaussian_eliminate(A_eq, b_eq)

# In toàn bộ 3 kết quả trả về
print_matrix("1. Ma trận mở rộng [A|b] sau khi khử Gauss (M_res)", M_res_eq)
print(f"2. Số lần hoán đổi dòng: {swaps_eq}")
print(f"3. Nghiệm của hệ x: {[round(val, 4) for val in sol_eq]}")

is_ok_eq = verify_solution(A_eq, b_eq, sol_eq)
print(f"=> Kiểm chứng NumPy: {'ĐÚNG' if is_ok_eq else 'SAI'}")

In [ ]:
print("\nKỊCH BẢN 2: HỆ VÔ SỐ NGHIỆM (TEST PIVOT BẰNG 0)")
A_inf = [[1, 2, 3], 
         [2, 4, 6], 
         [3, 6, 9]]
b_inf = [2, 4, 6]

print_matrix("Ma trận hệ số gốc: ", A_inf)
print(f"Vector vế phải: {b_inf}")
M_res_inf, sol_inf, swaps_inf = gaussian_eliminate(A_inf, b_inf)

print_matrix("1. Ma trận [A|b] sau khử (M_res_inf)", M_res_inf)
print(f"2. Số lần hoán đổi dòng (swaps): {swaps_inf}")
print(f"3. Kết quả: {sol_inf}") 

is_ok_inf = verify_solution(A_inf, b_inf, sol_inf)
print(f"=> Kiểm chứng NumPy: {'ĐÚNG' if is_ok_inf else 'SAI'}")

In [ ]:
print("\nKỊCH BẢN 3: HỆ VÔ NGHIỆM")
A_no = [[1, 2, 3], 
         [2, 4, 6], 
         [3, 6, 9]]
b_no = [2, 4, 100] 
print_matrix("Ma trận hệ số gốc", A_no)
print(f"Vector vế phải: {b_no}")

try:
    gaussian_eliminate(A_no, b_no)
except ValueError as e:
    print(f"1. Thuật toán đã dừng và ném ra lỗi: '{e}'")
    print("2. Quá trình khử bị hủy do phát hiện mâu thuẫn toán học.")

is_ok_no = verify_solution(A_no, b_no, None)
print(f"=> Kiểm chứng NumPy: {'ĐÚNG' if is_ok_no else 'SAI'}")

In [ ]:
print("\nKỊCH BẢN 4: TEST PARTIAL PIVOTING (PIVOT CỰC NHỎ)")
A_tiny = [[1e-14, 1, 1], 
          [1, -1, 2], 
          [2, 1, -1]]
b_tiny = [2, 2, 2]

print_matrix("Ma trận hệ số gốc A_tiny", A_tiny)
print("\n>> Đang thực hiện phép khử Gauss...")
M_res_t, sol_t, swaps_t = gaussian_eliminate(A_tiny, b_tiny)

print_matrix("1. Ma trận mở rộng sau khử", M_res_t)
print(f"2. Số lần hoán đổi dòng: {swaps_t} (Đã swap để đẩy 1e-14 xuống dưới!)")
print(f"3. Nghiệm x: {[round(val, 4) for val in sol_t]}")
print(f"=>Kiểm chứng NumPy: {'ĐÚNG' if verify_solution(A_tiny, b_tiny, sol_t) else 'SAI'}")

**2. Tính định thức ma trận**

In [62]:
def determinant(matrix_A):
    """
    Tính định thức của ma trận qua khử Gauss

    Args:
        A: Ma trận hệ số
    
    Return:
        Giá trị định thức
        Hoặc 0.0 nếu suy biến
        
    """
    if not matrix_A or not matrix_A[0]:
        return 0.0
    n = len(matrix_A)
    M = [row[:] for row in matrix_A]
    det = 1.0
    s = 0
    EPSILON = 1e-12

    for i in range(n):
        # 1. Tìm phần tử chốt (pivot) lớn nhất trên cột i
        pivot_row = i
        max_val = abs(M[i][i])
        for k in range(i + 1, n):
            if abs(M[k][i]) > max_val:
                max_val = abs(M[k][i])
                pivot_row = k
        
        # 2. Báo lỗi nếu cột toàn số 0 (ma trận suy biến)
        if max_val < EPSILON:
            print(f"không có pivot tại cột {i}")
            return 0.0
            
        # 3. Hoán đổi dòng và đổi dấu định thức nếu có đổi chỗ
        if pivot_row != i:
            M[i], M[pivot_row] = M[pivot_row], M[i]
            s += 1
            
        det *= M[i][i]
        
        # 4. Khử Gauss các phần tử bên dưới đường chéo chính
        for j in range(i + 1, n):
            factor = M[j][i] / M[i][i]
            for k in range(i + 1, n):
                M[j][k] -= factor * M[i][k]
                
    return ((-1) ** s) * det


def verify_determinant(matrix_A, custom_det):
    """
    Kiểm chứng kết quả định thức bằng NumPy

    Args:
        matrix_A: Ma trận hệ số
        custom_det: Giá trị định thức
    
    Returns:
        True: Nếu trùng khớp
        False: Nếu không trùng khớp
    """
    import numpy as np
    if not matrix_A:
        return custom_det == 0.0
    
    A_np = np.array(matrix_A, dtype=float)
    numpy_det = np.linalg.det(A_np)
    
    # Sử dụng isclose vì tính toán số thực luôn có sai số nhỏ
    return np.isclose(custom_det, numpy_det)

- Kiểm thử và kiểm chứng

In [ ]:
print("KỊCH BẢN 1: MA TRẬN BÌNH THƯỜNG (HAPPY PATH)")
print("-" * 60)
A_det = [[1, 2, 3], 
         [0, 1, 4], 
         [5, 6, 0]]

print_matrix("Ma trận A", A_det)
det_val = determinant(A_det)
print(f">> Kết quả Định thức: {det_val}")

is_ok_det = verify_determinant(A_det, det_val)
print(f"1. Kịch bản 1:     {'ĐÚNG' if is_ok_det else 'SAI'}")

# ------------------------------------------------------------------------------
print("\nKỊCH BẢN 2: TEST PARTIAL PIVOTING VÀ ĐẢO DẤU")
A_swap = [[1e-14, 1, 1], 
          [1, -1, 2], 
          [2, 1, -1]]

print_matrix("Ma trận cần Swap A_swap", A_swap)
det_swap = determinant(A_swap)
print(f">> Kết quả Định thức: {det_swap}")

is_ok_swap = verify_determinant(A_swap, det_swap)
print(f"3. Kịch bản 2 (Đảo dấu Swap):   {'ĐÚNG' if is_ok_swap else 'SAI'}")

**3. Tìm ma trận nghịch đảo** 

In [64]:
def inverse(matrix_A):
    """
    Tính ma trận nghịch đảo A^-1 bằng phương pháp Gauss-Jordan có Partial Pivoting.

    Args:
        matrix_A: Ma trận hệ số
    
    Returns:
        Ma trận nghịch đảo

    """
    n = len(matrix_A)
    # 1. Kiểm tra ma trận có vuông không
    for row in matrix_A:
        if len(row) != n:
            raise ValueError("Ma trận không vuông, không thể tìm nghịch đảo.")
    # 2. Tạo ma trận  M = [A | I]
    M = []
    for i in range(n):
        row_A = [element for element in matrix_A[i]]
        row_I=[1.0 if i==j else 0.0 for j in range(n) ]
        M.append(row_A + row_I)

    EPSILON = 1e-12 # Ngưỡng để kiểm tra số 0, tránh sai số float
    # 3. Quá trình khử Gauss-Jordan
    for k in range(n):
        #a. Tìm dòng p có phần tử chốt lớn nhất từ dòng k trở xuống
        p = k
        for i in range(k + 1, n):
            if abs(M[i][k]) > abs(M[p][k]):
                p=i
        if abs(M[p][k]) < EPSILON:
            print(f"Không có pivot tại cột {k}")
            return None
        # b. Hoán đổi dòng p và dòng k nếu cần
        if p != k:
            M[k], M[p] = M[p], M[k]
        # c. Chuẩn hóa dòng k
        pivot_val = M[k][k]
        for j in range(k, 2 * n):
            M[k][j] /= pivot_val
        # d. Khử Gauss-Jordan
        for i in range(n):
            if i != k:
                factor = M[i][k]
                for j in range(k, 2 * n):
                    M[i][j]=M[i][j]-factor*M[k][j]
    #4. Ma trận nghịch đảo
    inverse_matrix = []
    for i in range(n):
        inverse_matrix.append(M[i][n:])
    return inverse_matrix


def verify_inverse(matrix_A, inverse_A):
    """
    Kiểm chứng AA^-1 = I và so sánh kết quả với NumPy.

    Args:
        matrix_A: Ma trận hệ số
        inverse_A: Ma trận nghịch đảo
    
    Returns:
        True: Nếu tích AA^{-1} ra đúng ma trận đơn vị
        False: Nếu tích ra sai
    """
    import numpy as np
    # Kiểm tra xem inverse_A có tồn tại không (tránh lỗi khi hàm inverse tạch)
    if inverse_A is None:
        det_A = np.linalg.det(np.array(matrix_A, dtype=float))
        if abs(det_A) < 1e-9: 
            return True 
        else:
            return False 
        
    # 1. Kiểm tra kích thước trước khi tính toán để tránh lỗi Broadcast
    rows_A = len(matrix_A)
    cols_A = len(matrix_A[0])
    rows_inv = len(inverse_A)
    cols_inv = len(inverse_A[0])

    # Ma trận nghịch đảo phải vuông và cùng kích thước với ma trận gốc
    if rows_A != cols_A or rows_inv != cols_inv or rows_A != rows_inv:
        return False
    
    A_np = np.array(matrix_A, dtype=float)
    inv_custom_np = np.array(inverse_A, dtype=float)
    n = len(matrix_A)

    # 1. Kiểm tra điều kiện AA^-1 = I
    # Tính tích A * A^-1
    identity_check = np.dot(A_np, inv_custom_np)
    I_matrix = np.eye(n)
    
    # Kiểm tra xem tích có xấp xỉ ma trận đơn vị không
    is_identity = np.allclose(identity_check, I_matrix, atol=1e-8)

    # 2. So sánh trực tiếp với kết quả của NumPy
    try:
        inv_numpy = np.linalg.inv(A_np)
        matches_numpy = np.allclose(inv_custom_np, inv_numpy, atol=1e-8)
    except np.linalg.LinAlgError:
        matches_numpy = False

    return is_identity and matches_numpy

- Kiểm thử và kiểm chứng

In [ ]:
print("KỊCH BẢN 1: MA TRẬN KHẢ NGHỊCH")
print("-" * 60)
A_inv_good = [[1, 2, 3], 
              [0, 1, 4], 
              [5, 6, 0]]

print_matrix("Ma trận gốc A", A_inv_good)
print("\n>> Đang tìm ma trận nghịch đảo...")
inv_res_good = inverse(A_inv_good)

is_ok_inv_good = verify_inverse(A_inv_good, inv_res_good)
print(f"Kịch bản 1:     {'ĐÚNG' if is_ok_inv_good else 'SAI'}")
print_matrix("Ma trận nghịch đảo A^-1", inv_res_good)
# ------------------------------------------------------------------------------
print("KỊCH BẢN 2: MA TRẬN SUY BIẾN (KHÔNG KHẢ NGHỊCH)")

A_inv_zero = [[1, 2, 3], 
              [4, 5, 6], 
              [5, 7, 9]]

print_matrix("Ma trận suy biến A_zero", A_inv_zero)
print(">> Đang tìm ma trận nghịch đảo (Hãy chú ý thông báo lỗi Pivot)...")

inv_res_zero = inverse(A_inv_zero)
is_ok_inv_zero = verify_inverse(A_inv_zero, inv_res_zero)
print(f"Kịch bản 2:       {' ĐÚNG' if is_ok_inv_zero else '❌ SAI'}")

**4. Hạng và cơ sở của ma trận** 

In [69]:
def rank_and_basis(matrix_A):
    """
    Tính hạng và tìm cơ sở của không gian dòng, không gian cột, 
    và không gian nghiệm dựa trên dạng bậc thang rút gọn (RREF).

    Args:
        A: Ma trận hệ số
    
    Returns:
        Hạng ma trận, cơ sở không gian dòng, không gian cột, 
        và không gian nghiệm

    """
    if not matrix_A or not matrix_A[0]:
        return 0, [], [], []            
    
    rows = len(matrix_A)
    cols = len(matrix_A[0])
    # 1. Giữ lại bản sao của A gốc để tìm Không gian cột
    A_original= [[val for val in row] for row in matrix_A]

    # 2. Tạo ma trận M để khử Gauss-Jordan về RREF
    M = [[val for val in row] for row in matrix_A]
    EPSILON = 1e-12 # Ngưỡng để kiểm tra số 0, tránh sai số float
    pivot_row = 0
    pivot_cols = []

    # 3. Khử Gauss-Jordan về dạng RREF
    for j in range(cols):
        if pivot_row >= rows:
            break
        
        max_row = pivot_row            # Tìm phần tử chốt (trị tuyệt đối lớn nhất) để giảm sai số
        for i in range(pivot_row + 1, rows):
            if abs(M[i][j]) > abs(M[max_row][j]):
                max_row = i

        if abs(M[max_row][j]) < EPSILON:
            print(f"Không có pivot tại cột {j}")
            continue

        if max_row != pivot_row:
            M[pivot_row], M[max_row] = M[max_row], M[pivot_row]
        pivot_cols.append(j)
        pivot_val = M[pivot_row][j]
        for c in range(j, cols):
            M[pivot_row][c] /= pivot_val
        for i in range(rows):
            if i != pivot_row:
                factor = M[i][j]
                for c in range(j, cols):
                    M[i][c] -= factor * M[pivot_row][c]
        pivot_row += 1

    #4 Trích xuất dữ liệu
    # a. Hạng ma trận
    rank_matrix = len(pivot_cols)

    # b. Cơ sở Không gian dòng R(A)
    row_space_basis = [M[i] for i in range(rank_matrix)]

    # c. Cơ sở Không gian cột C(A)
    col_space_basis = []
    for j in pivot_cols:
        col = [A_original[i][j] for i in range(rows)]
        col_space_basis.append(col)

    # d. Cơ sở Không gian nghiệm N(A) (Tập nghiệm Ax = 0)
    null_space_basis = []
    free_cols = [j for j in range(cols) if j not in pivot_cols ]
    for free_col_idx in free_cols:
        x = [0.0]*cols
        x[free_col_idx] = 1.0
        for i in range(rank_matrix):
            p=pivot_cols[i]
            x[p]=-M[i][free_col_idx]
        null_space_basis.append(x)

    return rank_matrix, row_space_basis, col_space_basis, null_space_basis


def verify_rank_and_basis(A, rank_custom, row_basis, col_basis, null_basis):
    """
    Kiểm chứng  hạng và cơ sở của không gian dòng, không gian cột, 
    và không gian nghiệm bằng Numpy

    Args:
        A: Ma trận hệ số
        rank_custom: Hạng ma trận
        row_basis: cơ sở không gian dòng
        col_basis: Cơ sở không gian cột
        null_basis: Cơ sở không gian nghiệm
    
    Returns:
        True: Nếu trùng khớp
        False: Nếu không trùng khớp
    """
    import numpy as np
    A_np = np.array(A, dtype=float)
    rows, cols = A_np.shape

    # --- 1. Kiểm tra Hạng (Rank) ---
    rank_np = np.linalg.matrix_rank(A_np)
    check_rank = (rank_custom == rank_np)

    # --- 2. Kiểm tra Không gian dòng (Row Space) ---
    # Các vector trong row_basis phải độc lập tuyến tính và có số lượng = rank
    check_row = False
    if len(row_basis) == rank_custom:
        # Hạng của ma trận tạo bởi row_basis phải đúng bằng rank_custom
        if np.linalg.matrix_rank(np.array(row_basis)) == rank_custom:
            check_row = True

    # --- 3. Kiểm tra Không gian cột (Column Space) ---
    check_col = False
    if len(col_basis) == rank_custom:
        # Hạng của ma trận tạo bởi col_basis phải đúng bằng rank_custom
        if np.linalg.matrix_rank(np.array(col_basis).T) == rank_custom:
            check_col = True

    # --- 4. Kiểm tra Không gian nghiệm (Null Space) ---
    # A * v phải = 0 và số lượng vector = cols - rank
    check_null = True
    if len(null_basis) != (cols - rank_custom):
        check_null = False
    else:
        for v in null_basis:
            if not np.allclose(np.dot(A_np, np.array(v)), 0, atol=1e-10):
                check_null = False
                break

    return check_rank, check_row, check_col, check_null


- Kiểm thử và kiểm chứng

In [ ]:
print("KỊCH BẢN 1: MA TRẬN VUÔNG ĐẦY HẠNG")
A_r1 = [[1, 2, 3], 
        [0, 1, 4], 
        [5, 6, 0]]
print_matrix("Ma trận A (3x3)", A_r)

rank1,r_basis1, c_basis1, n_basis1  = rank_and_basis(A_r)
print(f">> Hạng của hệ vector: {rank1}")
print_matrix("Cơ sở không gian dòng ", r_basis1)
print("\nCơ sở không gian cột")
if c_basis1:
    for row in zip(*c_basis1):
        print("   " + "  ".join(f"[{val:>8}]" for val in row))
else:
    print("   (Trống)")
print_matrix("Cơ sở không gian nghiệm ", n_basis1)
print(f"=> Kiểm chứng: {'ĐÚNG' if verify_rank_and_basis(A_r1, rank1, r_basis1, c_basis1, n_basis1) else 'SAI'}")




In [ ]:
print("\nKỊCH BẢN 2: MA TRẬN CHỮ NHẬT")
A_r2 = [[1, 2, -1, 3], 
          [2, 4, -2, 6], 
          [3, 6, -3, 9]]

print_matrix("Ma trận A (3x4)", A_r2)
rank2, r_basis2, c_basis2, n_basis2 = rank_and_basis(A_r2)

print(f">> Hạng: {rank2}")
print_matrix("Cơ sở không gian dòng ", r_basis2)
print("\nCơ sở không gian cột")
if c_basis2:
    for row in zip(*c_basis2):
        print("   " + "  ".join(f"[{val:>8}]" for val in row))
else:
    print("   (Trống)")
print_matrix("Cơ sở không gian nghiệm ", n_basis2)

is_perfect_re = all(verify_rank_and_basis(A_r2, rank2, r_basis2, c_basis2, n_basis2))
print(f"=> Kiểm chứng tổng thể: {'ĐÚNG' if is_perfect_re else 'SAI'}")